In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score

In [10]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles in current directory:")
print(os.listdir())

Current directory:
/content

Files in current directory:
['.config', 'sample_data']


In [16]:
import pandas as pd
train = pd.read_csv("/content/dataset_C_training.csv")
test = pd.read_csv("/content/dataset_C_testing.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

Train Shape: (4756, 31)
Test Shape: (4749, 30)


In [17]:
train.head()

,respondent_id,covid_concern,covid_knowledge,behavioral_antiviral_meds,behavioral_avoidance,behavioral_face_mask,behavioral_wash_hands,behavioral_large_gatherings,behavioral_outside_home,behavioral_touch_face,...,employment_status,census_msa,household_adults,household_children,doctor_recc_covid,opinion_covid_vacc_effective,opinion_covid_risk,opinion_covid_sick_from_vacc,employment_sector,covid_vaccine
0,1,3.0,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,...,Employed,"MSA, Principle City",3.0,2.0,0,4,4,2.0,construction,0
1,2,2.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Employed,Non-MSA,0.0,0.0,0,5,2,1.0,education,1
2,3,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,2,2,5.0,wholesale,0
3,4,2.0,2.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,...,Not in Labor Force,"MSA, Not Principle City",1.0,0.0,1,3,3,2.0,NaN,1
4,5,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,Employed,"MSA, Not Principle City",0.0,0.0,0,3,2,2.0,wholesale,0


In [18]:
missing = train.isnull().sum()

missing[missing > 0].sort_values(ascending=False)

,0
employment_sector,2330
health_insurance,1908
income_poverty,748
rent_or_own,329
employment_status,238
marital_status,231
education,228
chronic_med_condition,169
child_under_6_months,135
health_worker,132


In [23]:
X = train.drop("covid_vaccine", axis=1)
y = train["covid_vaccine"]

print(X.shape)
print(y.shape)

(4756, 30)
(4756,)


In [24]:
for col in X.columns:

    if X[col].dtype == "object":

        X[col] = X[col].fillna("Unknown")
        test[col] = test[col].fillna("Unknown")

    else:

        median_value = X[col].median()

        X[col] = X[col].fillna(median_value)
        test[col] = test[col].fillna(median_value)

In [25]:
categorical_cols = X.select_dtypes(include=["object"]).columns

print("Categorical Columns:")
print(categorical_cols)

Categorical Columns:
Index(['age_group', 'education', 'race', 'sex', 'income_poverty',
       'marital_status', 'rent_or_own', 'employment_status', 'census_msa',
       'employment_sector'],
      dtype='object')


In [26]:
for col in categorical_cols:

    encoder = LabelEncoder()

    combined = pd.concat([
        X[col].astype(str),
        test[col].astype(str)
    ])

    encoder.fit(combined)

    X[col] = encoder.transform(X[col].astype(str))
    test[col] = encoder.transform(test[col].astype(str))

In [27]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_valid.shape)

(3804, 30)
(952, 30)


In [28]:
base_model = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

model = AdaBoostClassifier(
    estimator=base_model,
    n_estimators=500,
    learning_rate=0.05,
    random_state=42
)

In [29]:
model.fit(X_train, y_train)

AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=3,
                                                    random_state=42),
                   learning_rate=0.05, n_estimators=500, random_state=42)

In [30]:
valid_pred = model.predict_proba(X_valid)[:, 1]

auc = roc_auc_score(
    y_valid,
    valid_pred
)

print("Validation AUC:", auc)

Validation AUC: 0.8290828739258895


In [31]:
test_pred = model.predict_proba(test)[:, 1]

print(test_pred[:10])

[0.27200519 0.24024455 0.23952162 0.34411911 0.18956027 0.54453614
 0.45609631 0.68418555 0.72658257 0.37595938]


In [32]:
submission = pd.DataFrame({
    "respondent_id": test["respondent_id"],
    "covid_vaccine": test_pred
})

submission.head()

,respondent_id,covid_vaccine
0,4757,0.272005
1,4758,0.240245
2,4759,0.239522
3,4760,0.344119
4,4761,0.189560


In [33]:
submission.to_csv(
    "AdaBoostValues.csv",
    index=False
)

print("Saved Successfully")

Saved Successfully


In [34]:
submission = pd.read_csv("AdaBoostValues.csv")

print(submission.shape)
submission.head()

(4749, 2)


,respondent_id,covid_vaccine
0,4757,0.272005
1,4758,0.240245
2,4759,0.239522
3,4760,0.344119
4,4761,0.189560


In [35]:
from google.colab import files

files.download("AdaBoostValues.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>